In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [2]:
#Load the dataset
data=pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
#preprocess the data
# drop irrelevent features
data=data.drop(["RowNumber","CustomerId","Surname"],axis=1)
data

In [4]:
#Encode categorical variables
label_encoder_gender=LabelEncoder()
data["Gender"]=label_encoder_gender.fit_transform(data["Gender"])

In [5]:
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [6]:
#Onehot encode "Geography"
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo=OneHotEncoder()
geo_encoder=onehot_encoder_geo.fit_transform(data[["Geography"]])
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [7]:
geo_encoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [8]:
onehot_encoder_geo.get_feature_names_out(["Geography"])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [9]:
geo_encoded_df=pd.DataFrame(geo_encoder.toarray(),columns=onehot_encoder_geo.get_feature_names_out(["Geography"]))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [10]:
# Combine one hot encoder columns with the original data
data=pd.concat([data.drop("Geography", axis=1),geo_encoded_df],axis=1)

In [11]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [12]:
# Save the encoders as pickle files
with open("label_encoder_gender.pkl","wb") as file:
    pickle.dump(label_encoder_gender,file)
with open("onehot_encoder_geo.pkl","wb") as file:
    pickle.dump(onehot_encoder_geo,file)

In [13]:
#divide the dataset into independent and dependent features
X=data.drop("Exited",axis=1)
y=data["Exited"]

# Split the data into training and test data
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.8,random_state=42)

#Scale the data
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [14]:
# Save the scaler as pickle file
with open("scaler.pkl","wb") as file:
    pickle.dump(scaler,file)

#ANN Implementation

In [15]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [16]:
# Build our ANN Model
model=Sequential([
    Dense(64,activation="relu",input_shape=(X_train.shape[1],)), # HL1 connected with input layer
    Dense(32,activation="relu"),# HL2 connected with HL1
    Dense(1,activation="sigmoid") # Output layer connected with HL2
])

In [17]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2,945
Trainable params: 2,945
Non-trainable params: 0
_________________________________________________________________


In [18]:
# In order to do forward and backward propagation, we need to compile the model.
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01) 
los=tensorflow.keras.losses.BinaryCrossentropy()

model.compile(optimizer=opt,loss=los,metrics=["accuracy"])

In [19]:
# set up the Tensor board,Earlystopping
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard

# set up the Tensor board
log_dir="logs/fit/"+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)


In [20]:
# Set up EarlyStopping
earlystopping_callback=EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True)

In [21]:
# Train the model
history= model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,callbacks=[tensorboard_callback,earlystopping_callback])

Epoch 1/100
63/63 [==============================] - 4s 26ms/step - loss: 0.4593 - accuracy: 0.8070 - val_loss: 0.4224 - val_accuracy: 0.8244
Epoch 2/100
63/63 [==============================] - 1s 15ms/step - loss: 0.3895 - accuracy: 0.8360 - val_loss: 0.3880 - val_accuracy: 0.8356
Epoch 3/100
63/63 [==============================] - 1s 14ms/step - loss: 0.3612 - accuracy: 0.8475 - val_loss: 0.3683 - val_accuracy: 0.8489
Epoch 4/100
63/63 [==============================] - 1s 12ms/step - loss: 0.3537 - accuracy: 0.8560 - val_loss: 0.3651 - val_accuracy: 0.8468
Epoch 5/100
63/63 [==============================] - 1s 21ms/step - loss: 0.3408 - accuracy: 0.8620 - val_loss: 0.3664 - val_accuracy: 0.8462
Epoch 6/100
63/63 [==============================] - 1s 14ms/step - loss: 0.3390 - accuracy: 0.8645 - val_loss: 0.3686 - val_accuracy: 0.8514
Epoch 7/100
63/63 [==============================] - 1s 13ms/step - loss: 0.3243 - accuracy: 0.8625 - val_loss: 0.3750 - val_accuracy: 0.8486
Epoch 

In [22]:
model.save("model.h5")

In [23]:
# load tensor board extension
import pkg_resources

ModuleNotFoundError: No module named 'pkg_resources'

In [ ]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [ ]:
%tensorboard --logdir logs/fit20260417-145320

Reusing TensorBoard on port 6006 (pid 14456), started 3 days, 2:44:09 ago. (Use '!kill 14456' to kill it.)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential

print("Keras working ✅")

ModuleNotFoundError: No module named 'tensorflow.keras'